# Flask 웹에서 자동 수집 → 전처리 → 학습 → 시각화까지 자동화
```
전체 구조 흐름 (파이프라인)
csharp
복사
편집
[1] API 자동 수집 (1시간마다)
     ↓
[2] CSV 저장 (갱신)
     ↓
[3] 전처리 (장마/건기 분리 + 결측 제거 등)
     ↓
[4] 모델 훈련 (분류/회귀 등)
     ↓
[5] 예측/시각화 결과 생성
     ↓
[6] Flask 웹에 반영 (예: 서울 25개구 침수 위험 등)


--------------------------------------------------------------

구현 방식 요약
단계	구현 예	자동화 방법
1. 데이터 수집	schedule or cron	1시간마다 API로 자동 수집 (asos_collector.py)
2. 전처리	pandas 스크립트 함수화	수집 직후 rainy.csv, dry.csv 자동 생성
3. 학습/모델 저장	scikit-learn, joblib	수집 + 전처리 끝나면 자동 훈련 후 .pkl 저장
4. 시각화	matplotlib, plotly, seaborn, folium 등	모델 성능, 위험 지도, 추세 그래프 생성
5. Flask 연결	/api/update, /api/predict 등	사용자 요청 시 최신 모델 결과 제공

--------------------------------------------------------------

Flask에서 구성할 수 있는 주요 API 예
경로	기능
/api/auto_update	🔁 수집 + 전처리 + 모델 학습 자동 실행
/api/visualize	📊 성능/침수 위험 지도 등 JSON or 이미지 반환
/api/predict	🌧 특정 날짜/시간에 침수 위험 예측 반환
/dashboard	🖥 웹 UI 페이지 (시각화 포함)


백엔드 자동화 흐름 (예시)
python
복사
편집
@app.route('/api/auto_update', methods=['POST'])
def auto_update():
    # 1. 수집
    job()  # asos_collector.py에 있는 수집 함수

    # 2. 전처리
    preprocess_data()  # rainy.csv, dry.csv 생성

    # 3. 모델 훈련
    model = train_model('rainy.csv')  # 예: RandomForestClassifier
    joblib.dump(model, 'rainy_model.pkl')

    # 4. 시각화
    generate_performance_plot(model)
    generate_flood_risk_map()

    return jsonify({'status': '완료', 'message': '모든 단계 자동 실행 완료'})
    
-----------------------------------------------------------------------------------

가능 시각화 예
시각화 종류	라이브러리
침수 위험 지도	folium, plotly, seaborn, matplotlib
모델 성능 그래프	matplotlib, seaborn
시간별 강수량 그래프	plotly interactive
모델 비교 막대차트	/api/model_compare로 구현 가능
    
필요한 요소들
기술 요소	설명
Flask	백엔드 서버
joblib or pickle	모델 저장 및 재사용
schedule, threading, APScheduler	백그라운드 자동 스케줄링
matplotlib, plotly, folium	시각화
pandas, scikit-learn	전처리 및 모델 훈련

------------------------------------------------------------------------------------

최종 목표: “자동 침수 예측 웹 시스템”
서울 25개구 실시간 위험 예측

지도 기반 시각화 + 레벨 표시

자동 학습 및 업데이트

모델 성능 모니터링 대시보드

```

In [1]:
pip install requests pandas schedule

Note: you may need to restart the kernel to use updated packages.


# api key 활용 및 데이터 수집

In [4]:
from dotenv import load_dotenv
OPENWEATHER_API_KEY = load_dotenv('.env')
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

# ASOS 시간별 자료 수집

In [8]:
import os
import time
import requests
import pandas as pd
import datetime
import schedule

#  사용자 설정
SERVICE_KEY = OPENWEATHER_API_KEY
CSV_FILE = 'asos_seoul_hourly.csv'
STN_ID = 108
NUM_OF_ROWS = 800
API_URL = "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"

START_DATE = datetime.date(2020, 7, 10)
END_DATE = datetime.date.today() - datetime.timedelta(days=1)

def load_existing_times():
    if not os.path.exists(CSV_FILE):
        return set()
    df = pd.read_csv(CSV_FILE)
    return set(df['tm'].astype(str))

def fetch_day_data(date_obj, existing_times):
    """지정된 하루치(00~23시) 데이터를 수집"""
    all_results = []
    start_dt = datetime.datetime.combine(date_obj, datetime.time(0, 0))
    end_dt = datetime.datetime.combine(date_obj, datetime.time(23, 0))

    page = 1
    while True:
        params = {
            'serviceKey': SERVICE_KEY,
            'numOfRows': NUM_OF_ROWS,
            'pageNo': page,
            'dataCd': 'ASOS',
            'dateCd': 'HR',
            'startDt': start_dt.strftime("%Y%m%d"),
            'startHh': "00",
            'endDt': end_dt.strftime("%Y%m%d"),
            'endHh': "23",
            'stnIds': STN_ID,
            'dataType': 'JSON'
        }

        try:
            response = requests.get(API_URL, params=params, timeout=60)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f" [{date_obj}] 요청 실패: {e}")
            return []

        try:
            data = response.json()
            result_code = data['response']['header']['resultCode']
            result_msg = data['response']['header']['resultMsg']
            if result_code != '00':
                print(f" [{date_obj}] API 오류: {result_msg}")
                return []
        except Exception as e:
            print(f" [{date_obj}] JSON 파싱 실패")
            print(response.text[:300])
            return []

        items = data['response']['body']['items'].get('item', [])
        if not items:
            break

        for item in items:
            if item['tm'] not in existing_times:
                all_results.append(item)

        if len(items) < NUM_OF_ROWS:
            break
        page += 1

    print(f" [{date_obj}] {len(all_results)}건 수집 성공")
    return all_results

def append_to_csv(new_data):
    if not new_data:
        return
    df = pd.DataFrame(new_data)
    write_header = not os.path.exists(CSV_FILE)
    df.to_csv(CSV_FILE, mode='a', header=write_header, index=False)

def get_last_collected_date():
    if not os.path.exists(CSV_FILE):
        return START_DATE
    df = pd.read_csv(CSV_FILE)
    last_time = pd.to_datetime(df['tm']).max()
    return last_time.date() + datetime.timedelta(days=1)

def job():
    existing_times = load_existing_times()
    current_date = get_last_collected_date()

    while current_date <= END_DATE:
        new_data = fetch_day_data(current_date, existing_times)
        append_to_csv(new_data)
        current_date += datetime.timedelta(days=1)
        time.sleep(0.5)  # 과도한 요청 방지

# 최초 실행
job()

# 매시간마다 실행되도록 설정
schedule.every(1).hours.do(job)

print(" 자동 수집 대기 중...")
while True:
    schedule.run_pending()
    time.sleep(60)


✅ [2020-07-28] 24건 수집 성공
✅ [2020-07-29] 24건 수집 성공
✅ [2020-07-30] 24건 수집 성공
✅ [2020-07-31] 24건 수집 성공
✅ [2020-08-01] 24건 수집 성공
✅ [2020-08-02] 24건 수집 성공
✅ [2020-08-03] 24건 수집 성공
✅ [2020-08-04] 24건 수집 성공
✅ [2020-08-05] 24건 수집 성공
✅ [2020-08-06] 24건 수집 성공
✅ [2020-08-07] 24건 수집 성공
✅ [2020-08-08] 24건 수집 성공
✅ [2020-08-09] 24건 수집 성공
✅ [2020-08-10] 24건 수집 성공
✅ [2020-08-11] 24건 수집 성공
✅ [2020-08-12] 24건 수집 성공
✅ [2020-08-13] 24건 수집 성공
✅ [2020-08-14] 24건 수집 성공
✅ [2020-08-15] 24건 수집 성공
✅ [2020-08-16] 24건 수집 성공
✅ [2020-08-17] 24건 수집 성공
✅ [2020-08-18] 24건 수집 성공
✅ [2020-08-19] 24건 수집 성공
✅ [2020-08-20] 24건 수집 성공
✅ [2020-08-21] 24건 수집 성공
✅ [2020-08-22] 24건 수집 성공
✅ [2020-08-23] 24건 수집 성공
✅ [2020-08-24] 24건 수집 성공
✅ [2020-08-25] 24건 수집 성공
✅ [2020-08-26] 24건 수집 성공
✅ [2020-08-27] 24건 수집 성공
✅ [2020-08-28] 24건 수집 성공
✅ [2020-08-29] 24건 수집 성공
✅ [2020-08-30] 24건 수집 성공
✅ [2020-08-31] 24건 수집 성공
✅ [2020-09-01] 24건 수집 성공
✅ [2020-09-02] 24건 수집 성공
✅ [2020-09-03] 24건 수집 성공
✅ [2020-09-04] 24건 수집 성공
✅ [2020-09-05] 24건 수집 성공


✅ [2021-06-13] 24건 수집 성공
✅ [2021-06-14] 24건 수집 성공
✅ [2021-06-15] 24건 수집 성공
✅ [2021-06-16] 24건 수집 성공
✅ [2021-06-17] 24건 수집 성공
✅ [2021-06-18] 24건 수집 성공
✅ [2021-06-19] 24건 수집 성공
✅ [2021-06-20] 24건 수집 성공
✅ [2021-06-21] 24건 수집 성공
✅ [2021-06-22] 24건 수집 성공
✅ [2021-06-23] 24건 수집 성공
✅ [2021-06-24] 24건 수집 성공
✅ [2021-06-25] 24건 수집 성공
✅ [2021-06-26] 24건 수집 성공
✅ [2021-06-27] 24건 수집 성공
✅ [2021-06-28] 24건 수집 성공
✅ [2021-06-29] 24건 수집 성공
✅ [2021-06-30] 24건 수집 성공
✅ [2021-07-01] 24건 수집 성공
✅ [2021-07-02] 24건 수집 성공
✅ [2021-07-03] 24건 수집 성공
✅ [2021-07-04] 24건 수집 성공
✅ [2021-07-05] 24건 수집 성공
✅ [2021-07-06] 24건 수집 성공
✅ [2021-07-07] 24건 수집 성공
✅ [2021-07-08] 24건 수집 성공
✅ [2021-07-09] 24건 수집 성공
✅ [2021-07-10] 24건 수집 성공
✅ [2021-07-11] 24건 수집 성공
✅ [2021-07-12] 24건 수집 성공
✅ [2021-07-13] 24건 수집 성공
✅ [2021-07-14] 24건 수집 성공
✅ [2021-07-15] 24건 수집 성공
✅ [2021-07-16] 24건 수집 성공
✅ [2021-07-17] 24건 수집 성공
✅ [2021-07-18] 24건 수집 성공
✅ [2021-07-19] 24건 수집 성공
✅ [2021-07-20] 24건 수집 성공
✅ [2021-07-21] 24건 수집 성공
✅ [2021-07-22] 24건 수집 성공


✅ [2022-04-29] 24건 수집 성공
✅ [2022-04-30] 24건 수집 성공
✅ [2022-05-01] 24건 수집 성공
✅ [2022-05-02] 24건 수집 성공
✅ [2022-05-03] 24건 수집 성공
✅ [2022-05-04] 24건 수집 성공
✅ [2022-05-05] 24건 수집 성공
✅ [2022-05-06] 24건 수집 성공
✅ [2022-05-07] 24건 수집 성공
✅ [2022-05-08] 24건 수집 성공
✅ [2022-05-09] 24건 수집 성공
✅ [2022-05-10] 24건 수집 성공
✅ [2022-05-11] 24건 수집 성공
✅ [2022-05-12] 24건 수집 성공
✅ [2022-05-13] 24건 수집 성공
✅ [2022-05-14] 24건 수집 성공
✅ [2022-05-15] 24건 수집 성공
✅ [2022-05-16] 24건 수집 성공
✅ [2022-05-17] 24건 수집 성공
✅ [2022-05-18] 24건 수집 성공
✅ [2022-05-19] 24건 수집 성공
✅ [2022-05-20] 24건 수집 성공
✅ [2022-05-21] 24건 수집 성공
✅ [2022-05-22] 24건 수집 성공
✅ [2022-05-23] 24건 수집 성공
✅ [2022-05-24] 24건 수집 성공
✅ [2022-05-25] 24건 수집 성공
✅ [2022-05-26] 24건 수집 성공
✅ [2022-05-27] 24건 수집 성공
✅ [2022-05-28] 24건 수집 성공
✅ [2022-05-29] 24건 수집 성공
✅ [2022-05-30] 24건 수집 성공
✅ [2022-05-31] 24건 수집 성공
✅ [2022-06-01] 24건 수집 성공
✅ [2022-06-02] 24건 수집 성공
✅ [2022-06-03] 24건 수집 성공
✅ [2022-06-04] 24건 수집 성공
✅ [2022-06-05] 24건 수집 성공
✅ [2022-06-06] 24건 수집 성공
✅ [2022-06-07] 24건 수집 성공


✅ [2023-03-15] 24건 수집 성공
✅ [2023-03-16] 24건 수집 성공
✅ [2023-03-17] 24건 수집 성공
✅ [2023-03-18] 24건 수집 성공
✅ [2023-03-19] 24건 수집 성공
✅ [2023-03-20] 24건 수집 성공
✅ [2023-03-21] 24건 수집 성공
✅ [2023-03-22] 24건 수집 성공
✅ [2023-03-23] 24건 수집 성공
✅ [2023-03-24] 24건 수집 성공
✅ [2023-03-25] 24건 수집 성공
✅ [2023-03-26] 24건 수집 성공
✅ [2023-03-27] 24건 수집 성공
✅ [2023-03-28] 24건 수집 성공
✅ [2023-03-29] 24건 수집 성공
✅ [2023-03-30] 24건 수집 성공
✅ [2023-03-31] 24건 수집 성공
✅ [2023-04-01] 24건 수집 성공
✅ [2023-04-02] 24건 수집 성공
✅ [2023-04-03] 24건 수집 성공
✅ [2023-04-04] 24건 수집 성공
✅ [2023-04-05] 24건 수집 성공
✅ [2023-04-06] 24건 수집 성공
✅ [2023-04-07] 24건 수집 성공
✅ [2023-04-08] 24건 수집 성공
✅ [2023-04-09] 24건 수집 성공
✅ [2023-04-10] 24건 수집 성공
✅ [2023-04-11] 24건 수집 성공
✅ [2023-04-12] 24건 수집 성공
✅ [2023-04-13] 24건 수집 성공
✅ [2023-04-14] 24건 수집 성공
✅ [2023-04-15] 24건 수집 성공
✅ [2023-04-16] 24건 수집 성공
✅ [2023-04-17] 24건 수집 성공
✅ [2023-04-18] 24건 수집 성공
✅ [2023-04-19] 24건 수집 성공
✅ [2023-04-20] 24건 수집 성공
✅ [2023-04-21] 24건 수집 성공
✅ [2023-04-22] 24건 수집 성공
✅ [2023-04-23] 24건 수집 성공


✅ [2024-02-06] 24건 수집 성공
✅ [2024-02-07] 24건 수집 성공
✅ [2024-02-08] 24건 수집 성공
✅ [2024-02-09] 24건 수집 성공
✅ [2024-02-10] 24건 수집 성공
✅ [2024-02-11] 24건 수집 성공
✅ [2024-02-12] 24건 수집 성공
✅ [2024-02-13] 24건 수집 성공
✅ [2024-02-14] 24건 수집 성공
✅ [2024-02-15] 24건 수집 성공
✅ [2024-02-16] 24건 수집 성공
✅ [2024-02-17] 24건 수집 성공
✅ [2024-02-18] 24건 수집 성공
✅ [2024-02-19] 24건 수집 성공
✅ [2024-02-20] 24건 수집 성공
✅ [2024-02-21] 24건 수집 성공
✅ [2024-02-22] 24건 수집 성공
✅ [2024-02-23] 24건 수집 성공
✅ [2024-02-24] 24건 수집 성공
✅ [2024-02-25] 24건 수집 성공
✅ [2024-02-26] 24건 수집 성공
✅ [2024-02-27] 24건 수집 성공
✅ [2024-02-28] 24건 수집 성공
✅ [2024-02-29] 24건 수집 성공
✅ [2024-03-01] 24건 수집 성공
✅ [2024-03-02] 24건 수집 성공
✅ [2024-03-03] 24건 수집 성공
✅ [2024-03-04] 24건 수집 성공
✅ [2024-03-05] 24건 수집 성공
✅ [2024-03-06] 24건 수집 성공
✅ [2024-03-07] 24건 수집 성공
✅ [2024-03-08] 24건 수집 성공
✅ [2024-03-09] 24건 수집 성공
✅ [2024-03-10] 24건 수집 성공
✅ [2024-03-11] 24건 수집 성공
✅ [2024-03-12] 24건 수집 성공
✅ [2024-03-13] 24건 수집 성공
✅ [2024-03-14] 24건 수집 성공
✅ [2024-03-15] 24건 수집 성공
✅ [2024-03-16] 24건 수집 성공


✅ [2024-12-22] 24건 수집 성공
✅ [2024-12-23] 24건 수집 성공
✅ [2024-12-24] 24건 수집 성공
✅ [2024-12-25] 24건 수집 성공
✅ [2024-12-26] 24건 수집 성공
✅ [2024-12-27] 24건 수집 성공
✅ [2024-12-28] 24건 수집 성공
✅ [2024-12-29] 24건 수집 성공
✅ [2024-12-30] 24건 수집 성공
✅ [2024-12-31] 24건 수집 성공
✅ [2025-01-01] 24건 수집 성공
✅ [2025-01-02] 24건 수집 성공
✅ [2025-01-03] 24건 수집 성공
✅ [2025-01-04] 24건 수집 성공
✅ [2025-01-05] 24건 수집 성공
✅ [2025-01-06] 24건 수집 성공
✅ [2025-01-07] 24건 수집 성공
✅ [2025-01-08] 24건 수집 성공
✅ [2025-01-09] 24건 수집 성공
✅ [2025-01-10] 24건 수집 성공
✅ [2025-01-11] 24건 수집 성공
✅ [2025-01-12] 24건 수집 성공
✅ [2025-01-13] 24건 수집 성공
✅ [2025-01-14] 24건 수집 성공
✅ [2025-01-15] 24건 수집 성공
✅ [2025-01-16] 24건 수집 성공
✅ [2025-01-17] 24건 수집 성공
✅ [2025-01-18] 24건 수집 성공
✅ [2025-01-19] 24건 수집 성공
✅ [2025-01-20] 24건 수집 성공
✅ [2025-01-21] 24건 수집 성공
✅ [2025-01-22] 24건 수집 성공
✅ [2025-01-23] 24건 수집 성공
✅ [2025-01-24] 24건 수집 성공
✅ [2025-01-25] 24건 수집 성공
✅ [2025-01-26] 24건 수집 성공
✅ [2025-01-27] 24건 수집 성공
✅ [2025-01-28] 24건 수집 성공
✅ [2025-01-29] 24건 수집 성공
✅ [2025-01-30] 24건 수집 성공


KeyboardInterrupt: 

# 위 데이터 수집 코드 설명

In [ ]:
'''
핵심 요약
기능	설명
.env에서 API 키 불러오기	외부에 노출되지 않도록 보안 처리
하루 단위로 반복 수집	시간별 데이터 24건씩 수집
이미 수집된 데이터는 건너뜀	중복 저장 방지
API 오류/실패는 로그로 출력 후 무시	전체 흐름 끊기지 않게 설계
schedule로 1시간마다 자동 실행	실시간 운영 가능

------------------------------------------------------------------

# 필수 라이브러리 불러오기
import os
import time
import requests
import pandas as pd
import datetime
import schedule
from dotenv import load_dotenv

# .env 파일에서 환경변수(OPENWEATHER_API_KEY) 로드
OPENWEATHER_API_KEY = load_dotenv('.env')
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

#  사용자 설정
SERVICE_KEY = OPENWEATHER_API_KEY  # 공공데이터포털에서 발급받은 API 키
CSV_FILE = 'asos_seoul_hourly.csv'  # 수집된 데이터를 저장할 CSV 파일명
STN_ID = 108  # 서울 지역의 지점 코드
NUM_OF_ROWS = 800  # 한 페이지당 수집할 데이터 개수 (최대 1000까지 가능)
API_URL = "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"  # 기상청 ASOS API URL

# 수집 기간 설정
START_DATE = datetime.date(2020, 7, 10)
END_DATE = datetime.date(2025, 7, 9)

#  기존 CSV에서 이미 수집된 시간 데이터 불러오기
def load_existing_times():
    if not os.path.exists(CSV_FILE):
        return set()  # 파일이 없으면 빈 set 반환
    df = pd.read_csv(CSV_FILE)
    return set(df['tm'].astype(str))  # 'tm' 컬럼 값을 문자열로 변환해 set으로 반환

#  하루치 데이터(00~23시) 수집 함수
def fetch_day_data(date_obj, existing_times):
    all_results = []  # 최종 수집된 item 저장 리스트
    start_dt = datetime.datetime.combine(date_obj, datetime.time(0, 0))  # 시작 시각: 자정
    end_dt = datetime.datetime.combine(date_obj, datetime.time(23, 0))  # 종료 시각: 오후 11시

    page = 1  # 페이지네이션 시작
    while True:
        # API 요청 파라미터 구성
        params = {
            'serviceKey': SERVICE_KEY,
            'numOfRows': NUM_OF_ROWS,
            'pageNo': page,
            'dataCd': 'ASOS',
            'dateCd': 'HR',
            'startDt': start_dt.strftime("%Y%m%d"),
            'startHh': "00",
            'endDt': end_dt.strftime("%Y%m%d"),
            'endHh': "23",
            'stnIds': STN_ID,
            'dataType': 'JSON'
        }

        try:
            response = requests.get(API_URL, params=params, timeout=60)  # 60초 타임아웃 설정
            response.raise_for_status()  # HTTP 오류 발생 시 예외 발생
        except requests.exceptions.RequestException as e:
            print(f" [{date_obj}] 요청 실패: {e}")
            return []  # 실패한 경우 빈 리스트 반환하여 건너뜀

        try:
            data = response.json()  # JSON 응답 파싱
            result_code = data['response']['header']['resultCode']
            result_msg = data['response']['header']['resultMsg']
            if result_code != '00':
                print(f" [{date_obj}] API 오류: {result_msg}")
                return []  # API 응답이 정상 코드가 아닌 경우 무시
        except Exception as e:
            print(f" [{date_obj}] JSON 파싱 실패")
            print(response.text[:300])  # 에러 응답 일부 출력
            return []

        items = data['response']['body']['items'].get('item', [])  # 실제 관측 데이터 리스트
        if not items:
            break  # 데이터가 없으면 종료

        # 기존에 수집되지 않은 시간만 필터링
        for item in items:
            if item['tm'] not in existing_times:
                all_results.append(item)

        if len(items) < NUM_OF_ROWS:
            break  # 더 이상 다음 페이지가 없으면 종료
        page += 1  # 다음 페이지 요청

    print(f" [{date_obj}] {len(all_results)}건 수집 성공")
    return all_results

#  새로 수집된 데이터를 CSV에 저장
def append_to_csv(new_data):
    if not new_data:
        return
    df = pd.DataFrame(new_data)
    write_header = not os.path.exists(CSV_FILE)  # 헤더는 최초 1회만 기록
    df.to_csv(CSV_FILE, mode='a', header=write_header, index=False)

#  CSV에서 마지막 수집 날짜를 확인하여 이어서 수집
def get_last_collected_date():
    if not os.path.exists(CSV_FILE):
        return START_DATE  # 파일이 없으면 START_DATE부터 시작
    df = pd.read_csv(CSV_FILE)
    last_time = pd.to_datetime(df['tm']).max()  # 가장 마지막 관측 시각
    return last_time.date() + datetime.timedelta(days=1)  # 다음 날부터 수집

#  자동 수집 작업 정의
def job():
    existing_times = load_existing_times()  # 기존 수집 시간 목록
    current_date = get_last_collected_date()  # 다음 수집 시작 날짜

    while current_date <= END_DATE:
        new_data = fetch_day_data(current_date, existing_times)  # 하루치 수집
        append_to_csv(new_data)  # CSV에 저장
        current_date += datetime.timedelta(days=1)  # 다음 날짜로 이동
        time.sleep(0.5)  # 너무 빠른 요청 방지 (0.5초 대기)

#  최초 수집 실행
job()

#  매 시간마다 job()을 자동 실행
schedule.every(1).hours.do(job)

#  무한 루프: 스케줄링된 작업 수행 대기
print(" 자동 수집 대기 중...")
while True:
    schedule.run_pending()  # 예약된 작업 실행
    time.sleep(60)  # 1분 단위로 체크
'''

In [ ]:
'''
기상청 ASOS API는 지점마다 과거 수집 가능 범위가 다름

서울(지점 108)은 2000년대 이후 대부분 커버 가능

확인된 예: 2000-01-01 ~ 현재까지 수집 가능함
→ 직접 startDt=20000101로 조회해보면 정상 응답 확인 가능
'''

# 전처리 
## 각 파일은 머신러닝 모델에 직접 학습시킬 수 있을 정도로 정제된 상태(기상청 data)

In [9]:
import pandas as pd

# 1. 데이터 로드
df = pd.read_csv('asos_seoul_hourly.csv')

# 2. 날짜 파싱
df['tm'] = pd.to_datetime(df['tm'], errors='coerce')
df['year'] = df['tm'].dt.year
df['month'] = df['tm'].dt.month
df['day'] = df['tm'].dt.day
df['hour'] = df['tm'].dt.hour

# 3. 열 선택 (모델 입력용 주요 특성)
features = ['tm', 'ta', 'rn', 'ws', 'wd', 'hm', 'pa']
df = df[features].dropna()

# 4. 장마철 / 건조기 분리
RAINY_MONTHS = [6, 7]
DRY_MONTHS = [12, 1, 2]

rainy_df = df[df['tm'].dt.month.isin(RAINY_MONTHS)].copy()
dry_df = df[df['tm'].dt.month.isin(DRY_MONTHS)].copy()

# 5. 저장
rainy_df.to_csv('rainy.csv', index=False)
dry_df.to_csv('dry.csv', index=False)

print(f" 장마철 데이터: {len(rainy_df)}행 저장 완료 (rainy.csv)")
print(f" 건조기 데이터: {len(dry_df)}행 저장 완료 (dry.csv)")


 장마철 데이터: 1458행 저장 완료 (rainy.csv)
 건조기 데이터: 496행 저장 완료 (dry.csv)


In [ ]:
'''
rn > 0만 있는 장마철 강수 집중 시기만 추출도 가능

feature scaling, label 추가까지도 가능

계절별 모델 분기 훈련도 가능

이상치 제거 or 정규화 (StandardScaler, MinMaxScaler)

강수량 기준 이진 분류 레이블 생성 (flood = 1 if rn > threshold else 0)

장마철 전용 모델 vs 건조기 모델 성능 비교

-----------------------------------------------------------------------------

다음 추천 스텝
preprocess_data() 함수화 → Flask에 연결

train_model() 함수로 자동 학습 가능하도록 만들기

/api/auto_update API부터 만들어보자



'''